In [4]:
import os
import numpy as np
import scipy.stats as stats
import torch
import copy
from torch import nn
from torch import optim
import torch.nn.functional as F
import torchvision
from torchvision import datasets
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

In [5]:
with_cuda = torch.cuda.is_available()
if with_cuda:
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

# Text prediction

## The Penn-TreeBank (PTB) dataset

**Question 1**

Take a look at the files `ptb_test.txt`, `ptb_valid.txt`, `ptb_train.txt`. What are the differences with a real text? Why?

**Question 2**

We want to perform text prediction. For that, we have two choices: 
1. character prediction;
2. word prediction.

Specify the format of the input and the output of our model: how should we encode them?

On lit le texte séquentiellement et on attribut des chiffres aux mots que l'on rencontre. Pour les transformers, ce qui est utilisé est de contruire de manière automatique des tokens ou les choses sont découpées en fonction des groupes de lettre les p^lus fréquents. On spécifie un nombre de tokens. 

**Question 3**

We want to replace the characters/words by tokens. For that, we build the classes `Dictionary`, `CorpusWords` and `CorpusChars`.

The class `Dictionary` has attributes:
* a dictionary `word2idx`, which maps words (or characters) to tokens;
* a list `idx2word`, which maps tokens to words (or characters);
* a dictionary `nb_occ`, which contains the number of occurrences of each word (or character).

Write the method `add_word`, which takes a word (or character), and updates the attributes accordingly.

The class `CorpusWords` has attributes:
* the dictionary of words `dictionary`:
* the tokenized train, valid and test texts.

Write the method `tokenize`, which takes a path to a text file and:
1. reads it sequentially word by word and add the words to the dictionary;
2. reads it sequentially word by word, transforms the words into token (by using the dictionary), and store the tokens into a LongTensor; each "new line" character should also be stored;
3. returns the LongTensor in which the token have been stored.

Build the class `CorpusChars` in the same way (by replacing words by characters).

In [6]:
class Dictionary:
    def __init__(self):
        self.word2idx = {}
        self.idx2word = []
        self.nb_occ = {}

    def add_word(self, word):
        if word in self.word2idx.keys():
            self.nb_occ[word]+=1
        else : 
            self.nb_occ[word]=1
            self.word2idx[word]= len(self.word2idx)
            self.idx2word.append(word)
        
    def __len__(self):
        return len(self.idx2word)


class CorpusWords:
    def __init__(self, path):
        self.dictionary = Dictionary() #ce qui nous permet de passer des tokens aux mots et inversement
        self.train = self.tokenize(os.path.join(path, 'ptb_train.txt'))
        self.valid = self.tokenize(os.path.join(path, 'ptb_valid.txt'))
        self.test = self.tokenize(os.path.join(path, 'ptb_test.txt'))

    def tokenize(self, path):
        assert os.path.exists(path)
        with open(path, "r", encoding = "utf8") as f:
            tokens = 0
            for line in f: 
                words = line.split() + ['<eos>']
                tokens += len(words)
                for word in words :   
                        self.dictionary.add_word(word)
        
        with open(path, "r", encoding = "utf8") as f:
            token = 0
            ids = torch.LongTensor(tokens)
            for line in f: 
                words = line.split() + ['<eos>']

                for word in words :   
                        ids[token] = self.dictionary.word2idx[word]
                        token += 1
        return ids
#attention : pour utiliser la méthode add_word avec une variable de type dictionnaire, il faut procéder comme suit : dct.add_word(word), 
#puisque le self est implicit (il y a déjà deux arguments)        
    
class CorpusChars:
    def __init__(self, path):
        self.dictionary = Dictionary() #ce qui nous permet de passer des tokens aux mots et inversement
        self.train = self.tokenize(os.path.join(path, 'ptb_train.txt'))
        self.valid = self.tokenize(os.path.join(path, 'ptb_valid.txt'))
        self.test = self.tokenize(os.path.join(path, 'ptb_test.txt'))

    def tokenize(self, path):
        assert os.path.exists(path)
        with open(path, "r", encoding = "utf8") as f:
            tokens = 0
            for line in f: 
                words = list(line)
                tokens += len(words)
                for word in words :   
                        self.dictionary.add_word(word)
        
        with open(path, "r", encoding = "utf8") as f:
            token = 0
            ids = torch.LongTensor(tokens)
            for line in f: 
                words = list(line)

                for word in words :   
                        ids[token] = self.dictionary.word2idx[word]
                        token += 1
        return ids


In [33]:
import os
import sys

PARENT_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.append(PARENT_DIR)
print("Parent directory added to sys.path:", ".../" + os.path.basename(PARENT_DIR))

# Path to test data
DATA_TEST_PATH = PARENT_DIR + "/Deep_learning_3"+ "/TP_Deep_Learning" 
print(
    "Dataset directory used:",
    ".../" + os.path.basename(PARENT_DIR) + DATA_TEST_PATH.split(os.path.basename(PARENT_DIR))[-1]
)
print(os.getcwd())
C = CorpusChars(DATA_TEST_PATH)
W = CorpusWords(DATA_TEST_PATH)

Parent directory added to sys.path: .../L3
Dataset directory used: .../L3/Deep_learning_3/TP_Deep_Learning
c:\Users\camil\Desktop\L3\Deep_learning_3\TP_Deep_Learning


NB : si notre jeu de données est dans le même repository que notre code, pour appeler le jeu de donées, on peut marquer C = CorpusChars(".") et W = CorpusWords(".")
Ceci ouvre le dossier courant. 

In [8]:
dC = C.dictionary.word2idx
dW = W.dictionary.word2idx

dC2 = C.dictionary.idx2word
dW2 = W.dictionary.idx2word

On met $<eos>$ pour signifier la fin d'une phrase. Cela permet d'indiquer au RNN la fin de la phrase et de faire en sorte qu'il ne produise pas de sorties abérantes. 

**Question 4** 

What does the following functions do?

In [9]:
def batchify(data, batch_size):
    nbatches = data.size(0) // batch_size
    data = data.narrow(0, 0, nbatches * batch_size)
    data = data.view(batch_size, -1).t().contiguous()
    
    return data

In [10]:
B_words_train = batchify(W.train, 64)
B_chars_train = batchify(C.train, 64)

B_words_test = batchify(W.test, 64)
B_chars_test = batchify(C.test, 64)

In [11]:
def get_batch(data_src, bptt, i):
    seq_len = min(bptt, len(data_src) - 1 - i)
    data = data_src[i:i + seq_len]
    target = data_src[i + 1:i + 1 + seq_len].view(-1)
    
    return data, target

In [12]:
get_batch(B_words_train, 5, 2)

(tensor([[   2, 2756,  189,   42, 1639,   98, 3175, 4823, 2189,   64, 1621,  108,
          2716,  552,   35,  318, 8633,  590, 1933,   48,  307, 1616, 6265,  552,
          2173,  416, 1466,   26,   35, 3352, 3568, 6504, 4908, 1486,   93,  229,
          1655,   24,   26, 1623, 3730,   46, 2587,   24,  286,  321,  725,   65,
          5155, 4135,  424,   48, 2728,   24, 1515, 3049, 1602,   30,  850,   42,
          1760, 1168,  129, 1880],
         [   3, 2757,   30,  625,   64,   93,   64,  154,  853, 3989,   95, 7861,
          2648,  142, 1500, 5362,   24, 1120,  169, 2173,  245, 1617, 7316, 1480,
          3725,   27,   32,   54, 1746,   24, 1782,   26, 1153, 6220, 8436,   32,
            35, 4854,   54, 4190,  556,   98,  108,   32, 3990,   24,   27,   35,
           133,  718,  938, 3891,  113, 1635,   24, 1912,  195,   26,   42, 3692,
           108,   78,   24,  301],
         [   4,   39,  983,  337, 2984,  225,   32,  552, 6579,   32,  274,   40,
            32, 1983, 1501, 

Lorsque l'on fait batchify, on remplace un tenseur d'ordre 1 : $[1, 2, ...,  101,10,....]$ \
en un tableau (tenseur d'ordre 2): $[[1, 2, ... ], ...[101, 10,...], ...[...]]$

## Building a LSTM

**Question 5**

Build a class `ModelRNN`:
* which takes arguments:
  * `ntokens`: the number of different tokens (= the size of the dictionary);
  * `nhid`: the number of neurons in the hidden layer of the LSTM;
  * `nlayers`: the number of stacked LSTMs;
* which has attributes:
  * `rnn`: a `nn.LSTM` initialized with the relevant attributes;
  * exercise: other attributes the are necessary to make it work;
* which implements the method `forward`:
  * exercise: implement it; what should be the arguments and the return values of this function?

In [37]:
class ModelRNN(nn.Module):
    def __init__(self, ntokens, ninp, nhid, nlayers):
        super(ModelRNN, self).__init__()
        self.encoder = nn.Embedding(ntokens, ninp)
        self.rnn = nn.LSTM(ninp, nhid, nlayers)
        self.decoder = nn.Linear(nhid, ntokens)

        self.nhid = nhid
        self.nlayers = nlayers

    def forward(self, input, hidden):
        # input: (seq_len, batch_size)
        emb = self.encoder(input)  # (seq_len, batch_size, ninp)
        output, hidden = self.rnn(emb, hidden)
        decoded = self.decoder(output)  # (seq_len, batch_size, ntokens)
        return decoded, hidden

    def init_hidden(self, batch_size):
        device = next(self.parameters()).device
        return (torch.zeros((self.nlayers, batch_size, self.nhid), device=device),
                torch.zeros((self.nlayers, batch_size, self.nhid), device=device))

class torch.nn.Embedding(num_embeddings, embedding_dim, padding_idx=None, max_norm=None, norm_type=2.0, scale_grad_by_freq=False, sparse=False, _weight=None, _freeze=False, device=None, dtype=None)

**Question 6**

Complete the code in `train_model`

In [ ]:
def train_model(train_data, bptt, model, criterion, optimizer, nepochs, batch_size):
    train_losses = []

    for epoch in range(nepochs):
        model.train()
        hidden = model.init_hidden(batch_size)
        epoch_loss = 0.0

        print("Je mentraine")

        for batch_idx, i in enumerate(range(0, train_data.size(0) - 1, bptt)):
            print("batch courant =", batch_idx)
            data, targets = get_batch(train_data, bptt, i)
            data, targets = data.to(device), targets.to(device)

            # Detach hidden states to avoid backprop through entire history
            hidden = tuple(h.detach() for h in hidden)

            optimizer.zero_grad()
            output, hidden = model(data, hidden)

            # output: (seq_len, batch_size, ntokens) -> (seq_len*batch_size, ntokens)
            output_flat = output.view(-1, output.size(-1))
            targets_flat = targets.view(-1)

            loss = criterion(output_flat, targets_flat)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 0.25)
            optimizer.step()

            epoch_loss += loss.item()

        avg_loss = epoch_loss / (batch_idx + 1)
        train_losses.append(avg_loss)

        print('Epoch: {} \tTraining loss: {:.6f}'.format(epoch + 1, avg_loss))

    return train_losses

**Question 7**

Construct the batches for character prediction and train the model.

In [39]:
batch_size = 20
bptt = 35

ninp = 100
nhid = 100
nlayers = 2

ntokens = len(C.dictionary.word2idx)



In [42]:
model = ModelRNN(ntokens, ninp, nhid, nlayers)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr = .001)

nepochs = 5

train_model(train_data = batchify(C.train, batch_size), bptt = bptt, model = model, criterion = criterion, optimizer = optimizer, nepochs = nepochs, batch_size=batch_size)


Je mentraine
Epoch: 1 	Training loss: 1.529840
Je mentraine
Epoch: 2 	Training loss: 1.267541
Je mentraine
Epoch: 3 	Training loss: 1.206264
Je mentraine
Epoch: 4 	Training loss: 1.173034
Je mentraine
Epoch: 5 	Training loss: 1.151254


[1.5298400509337824,
 1.2675407873517726,
 1.2062639695082094,
 1.173033724157661,
 1.1512537086310948]

est-ce que l'on peut deviner combien de temps entrainer : propre à la taille du data set et plutôt de façon expértimentale. 

In [43]:
def evaluate_model(test_data, bptt, model, criterion, batch_size):
    model.eval()
    hidden = model.init_hidden(batch_size)
    total_loss = 0.0
    total_tokens = 0

    with torch.no_grad():
        for i in range(0, test_data.size(0) - 1, bptt):
            data, targets = get_batch(test_data, bptt, i)
            data, targets = data.to(device), targets.to(device)

            output, hidden = model(data, hidden)
            hidden = tuple(h.detach() for h in hidden)

            output_flat = output.view(-1, output.size(-1))
            targets_flat = targets.view(-1)

            loss = criterion(output_flat, targets_flat)
            total_loss += loss.item() * targets_flat.size(0)
            total_tokens += targets_flat.size(0)

    avg_loss = total_loss / total_tokens
    perplexity = torch.exp(torch.tensor(avg_loss))

    print('Test loss: {:.6f}'.format(avg_loss))
    print('Test perplexity: {:.6f}'.format(perplexity.item()))

    return avg_loss, perplexity.item()

# Évaluation du modèle de prédiction de caractères
test_data_chars = batchify(C.test, batch_size)
evaluate_model(test_data_chars, bptt, model, criterion, batch_size)

Test loss: 1.153808
Test perplexity: 3.170241


(1.1538075372089194, 3.170240640640259)

Tâche difficile => train et test loss similaires, on ne peut pas avoir d'avoir d'overfit car on peine à fiter

**Question 8**

Do the same for word prediction. What is the purpose of the `encoder` and the `decoder` modules?

## Rôle des modules `encoder` et `decoder`

### `encoder` (nn.Embedding)
- **Rôle** : Convertit les indices de tokens (entiers) en vecteurs denses de dimension `ninp`
- **Pourquoi ?** : Les tokens sont des indices discrets (0, 1, 2, ...), mais le RNN a besoin de représentations continues apprenables
- **Apprentissage** : Les embeddings s'apprennent pendant l'entraînement pour capturer la similarité sémantique entre tokens

### `decoder` (nn.Linear)
- **Rôle** : Convertit les états cachés du RNN (dimension `nhid`) en logits sur le vocabulaire (dimension `ntokens`)
- **Pourquoi ?** : Le RNN produit des représentations cachées, mais on veut prédire des probabilités sur tous les tokens possibles
- **Sortie** : Logits non-normalisés (passés ensuite à softmax via CrossEntropyLoss)

**En résumé** : L'encoder transforme l'entrée discrète en continue, le decoder transforme la sortie du RNN en prédictions de tokens.

In [ ]:
# Prédiction de mots (word prediction)

# Paramètres pour le modèle de mots
ntokens_words = len(W.dictionary.word2idx)
model_words = ModelRNN(ntokens_words, ninp, nhid, nlayers).to(device)

criterion_words = nn.CrossEntropyLoss()
optimizer_words = torch.optim.Adam(model_words.parameters(), lr=0.001)

# Entraînement du modèle de mots
train_data_words = batchify(W.train, batch_size)
train_losses_words = train_model(train_data_words, bptt, model_words, criterion_words, optimizer_words, nepochs, batch_size)

# Évaluation du modèle de mots
test_data_words = batchify(W.test, batch_size)
evaluate_model(test_data_words, bptt, model_words, criterion_words, batch_size)

Je mentraine
Epoch: 1 	Training loss: 6.217424
Je mentraine
Epoch: 2 	Training loss: 5.611594
Je mentraine
Epoch: 3 	Training loss: 5.356087
Je mentraine
